# Notebook 02: Statistical Hypothesis Testing, Target Leakage Proofs & Validation Design (Module 1)
**Project 02 — Enterprise Predictive Intelligence & Applied Machine Learning System**


In [1]:
import pandas as pd
import numpy as np
from scipy import stats

df_cust = pd.read_csv('../data/raw/customer_retention_features.csv')
g1 = df_cust[df_cust['no_purchase_next_90d'] == 1]
g0 = df_cust[df_cust['no_purchase_next_90d'] == 0]
for col in ['orders_last_180d', 'tenure_months', 'days_since_last_order', 'support_tickets_last_180d', 'discount_order_share', 'returns_last_180d', 'sessions_last_30d', 'avg_order_value']:
    u, p = stats.mannwhitneyu(g1[col], g0[col])
    r = df_cust[col].corr(df_cust['no_purchase_next_90d'])
    print(f'{col:26s} | r = {r:+.4f} | Mann-Whitney p = {p:.3e}')


orders_last_180d           | r = -0.2601 | Mann-Whitney p = 0.000e+00
tenure_months              | r = -0.2568 | Mann-Whitney p = 0.000e+00
days_since_last_order      | r = +0.2470 | Mann-Whitney p = 0.000e+00
support_tickets_last_180d  | r = +0.0859 | Mann-Whitney p = 1.567e-45
discount_order_share       | r = +0.0624 | Mann-Whitney p = 3.572e-27
returns_last_180d          | r = +0.0470 | Mann-Whitney p = 1.452e-13
sessions_last_30d          | r = -0.0364 | Mann-Whitney p = 3.986e-10
avg_order_value            | r = +0.0085 | Mann-Whitney p = 4.379e-01


## Deterministic Target Leakage Verification (`delivery_delay_training.csv`)
We prove mathematically that `delay_days` and `late_delivery` are exact deterministic functions of `actual_delivery_days` and `promised_delivery_days` across 100% of rows (`40,000 / 40,000`).


In [2]:
df_del = pd.read_csv('../data/raw/delivery_delay_training.csv')
calc_delay = (df_del['actual_delivery_days'] - df_del['promised_delivery_days']).clip(lower=0).round(4)
exact_delay_matches = (np.abs(calc_delay - df_del['delay_days']) < 1e-4).sum()
exact_late_matches = ((df_del['delay_days'] > 1.0) == (df_del['late_delivery'] == 1)).sum()
print(f'Exact formula match for delay_days: {exact_delay_matches} / {len(df_del)} (100.0%)')
print(f'Exact threshold match for late_delivery (delay_days > 1.0): {exact_late_matches} / {len(df_del)} (100.0%)')


Exact formula match for delay_days: 40000 / 40000 (100.0%)
Exact threshold match for late_delivery (delay_days > 1.0): 40000 / 40000 (100.0%)
